# Lab: temperature profiles and the basal thermal state

```{note}
This lab accompanies {doc}`thermal-structure`. It turns the heat budget of that chapter into working code, so read the chapter first and keep its equations in view.
```

In this lab you will build the vertical temperature profile of a glacier from the ground up, using nothing but numpy and matplotlib. By the end you should be able to

- construct the steady conduction-only temperature profile and read the basal temperature off it,
- add vertical advection and explain how the Peclet number reshapes the profile,
- propagate a periodic surface temperature into the ice and measure its e-folding depth,
- step a transient finite-difference solver forward in time and watch a climate signal diffuse into the ice,
- decide, for realistic parameter sets, whether a bed is frozen or thawed.

The question behind all of it is the one the chapter ends on. Is the bed at the pressure-melting point, where water lubricates the interface and sliding becomes possible, or is it frozen fast to the rock?


## The reduced heat equation

The full heat equation of the chapter balances storage, advection, conduction, and strain heating. Here we strip it down to a vertical column far from the margin. Let $z$ run upward from the bed, with the bed at $z=0$ and the surface at $z=H$. In steady state, with only vertical motion and no strain heating, what remains is a competition between vertical advection and vertical conduction,

$$ w\,\frac{dT}{dz} = \kappa\,\frac{d^2T}{dz^2}, $$

where $w(z)$ is the vertical velocity of the ice and $\kappa = k/\rho c$ is the thermal diffusivity. Two boundary conditions close the problem. At the surface the temperature is pinned to the mean annual air temperature, $T(H) = T_s$. At a frozen bed the geothermal flux $G$ has nowhere to go but up through the ice by conduction, which fixes the gradient there,

$$ \left.\frac{dT}{dz}\right|_{z=0} = -\frac{G}{k}. $$

The minus sign carries real physics. Heat flows from warm to cold, so for geothermal heat to move upward the temperature must fall with height above the bed. We will work in meters, years, and degrees Celsius throughout, which keeps every number in the code at a human scale.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Material properties of ice, as discussed in the chapter
rho = 917.0    # density, kg m^-3
c = 2097.0     # specific heat capacity, J kg^-1 K^-1
k = 2.10       # thermal conductivity, W m^-1 K^-1

SECONDS_PER_YEAR = 3.15576e7
kappa = k / (rho * c)                  # thermal diffusivity, m^2 s^-1
kappa_yr = kappa * SECONDS_PER_YEAR    # the same diffusivity in m^2 yr^-1

print(f'kappa = {kappa:.2e} m^2/s = {kappa_yr:.1f} m^2/yr')


def T_pmp(H):
    """Pressure-melting point (deg C) at the bed of ice H meters thick.

    The Clausius-Clapeyron slope for glacier ice is 0.07 to 0.10 K per
    100 m of overburden; we adopt a middle value of 0.087 K per 100 m.
    """
    return -8.7e-4 * H


print(f'Pressure-melting point under 3000 m of ice: {T_pmp(3000.0):.2f} C')


## Conduction only

Set $w = 0$ and the steady equation collapses to $d^2T/dz^2 = 0$, a uniform gradient. Integrating from the bed with the geothermal boundary condition and anchoring the surface at $T_s$ gives a straight line,

$$ T(z) = T_s + \frac{G}{k}\,(H - z). $$

A typical continental geothermal flux of $60\ \mathrm{mW\,m^{-2}}$ through ice with $k = 2.1\ \mathrm{W\,m^{-1}\,K^{-1}}$ produces a gradient near $29\ \mathrm{K\,km^{-1}}$, so every kilometer of ice buys the bed about 29 degrees of warming over the surface. The basal temperature is just $T_s + GH/k$, and the whole basal thermal state hinges on whether that number reaches the pressure-melting point.

**Task 1:** Complete `conduction_profile` below. Plot the profile for an ice-sheet interior with $T_s = -50$ °C, $H = 3000$ m, and $G = 50\ \mathrm{mW\,m^{-2}}$, mark the pressure-melting point at the bed, and state whether this bed is frozen or thawed. Then find, by trying a few values, roughly how thick the ice would need to be for the bed to reach the melting point with these values of $T_s$ and $G$.


In [ ]:
def conduction_profile(z, Ts, G, H):
    """Steady conduction-only temperature profile, deg C.

    z  : array of heights above the bed, m
    Ts : surface temperature, deg C
    G  : geothermal flux, W m^-2
    H  : ice thickness, m
    """
    # YOUR CODE HERE
    raise NotImplementedError


H = 3000.0
Ts = -50.0
G = 0.050
z = np.linspace(0.0, H, 301)

# T = conduction_profile(z, Ts, G, H)

fig, ax = plt.subplots(figsize=(5, 6))
# ax.plot(T, z, lw=2)
# ax.plot(T_pmp(H), 0.0, 'r*', ms=14, label='pressure-melting point')
ax.set_xlabel('temperature (deg C)')
ax.set_ylabel('height above bed (m)')
ax.legend()
plt.show()

# print(f'basal temperature: {T[0]:.1f} C   melting point: {T_pmp(H):.1f} C')


## Adding vertical advection

Snow falling on the surface buries the ice beneath it, so in steady state the whole column moves slowly downward. The simplest kinematics consistent with an accumulation rate $\dot b$ at the surface and no melting at the bed is a vertical velocity that shrinks linearly with depth,

$$ w(z) = -\dot b\,\frac{z}{H}, $$

downward everywhere, largest at the surface, zero at the bed. Substituting into the steady balance turns it into a first-order equation for the gradient $\theta = dT/dz$,

$$ \frac{d\theta}{dz} = \frac{w(z)}{\kappa}\,\theta \qquad\Longrightarrow\qquad \theta(z) = -\frac{G}{k}\,\exp\!\left[-\frac{\mathrm{Pe}}{2}\left(\frac{z}{H}\right)^2\right], $$

where the dimensionless group

$$ \mathrm{Pe} = \frac{\dot b\, H}{\kappa} $$

is the Peclet number, the ratio of the conduction time $H^2/\kappa$ to the advection time $H/\dot b$. This is the classic solution of Robin (1955). Integrating the gradient once more gives the temperature,

$$ T(z) = T_s + \frac{G}{k}\int_z^H \exp\!\left[-\frac{\mathrm{Pe}}{2}\left(\frac{\zeta}{H}\right)^2\right] d\zeta, $$

an integral with no elementary closed form, which is no obstacle at all. Evaluate the gradient on your grid, accumulate it numerically with a trapezoid rule, and shift the result so the surface sits at $T_s$.

**Task 2:** Complete `robin_profile` below. With $T_s = -50$ °C, $H = 3000$ m, and $G = 50\ \mathrm{mW\,m^{-2}}$ held fixed, plot profiles for $\dot b = 0, 0.05, 0.1, 0.3,$ and $1.0\ \mathrm{m\,yr^{-1}}$, and print the Peclet number and basal temperature for each. Check that the $\dot b = 0$ case reproduces your Task 1 line.


In [ ]:
def robin_profile(z, Ts, G, H, bdot):
    """Steady advection-diffusion (Robin) temperature profile, deg C.

    z    : array of heights above the bed, m (must start at 0)
    Ts   : surface temperature, deg C
    G    : geothermal flux, W m^-2
    H    : ice thickness, m
    bdot : surface accumulation rate, m yr^-1 (ice equivalent)
    """
    Pe = bdot * H / kappa_yr
    dTdz = -(G / k) * np.exp(-0.5 * Pe * (z / H)**2)
    # Integrate dTdz upward from the bed (a cumulative trapezoid works:
    # accumulate 0.5*(dTdz[1:] + dTdz[:-1])*dz with np.cumsum, with a
    # zero prepended), then add a constant so that T at z = H equals Ts.
    # YOUR CODE HERE
    raise NotImplementedError


fig, ax = plt.subplots(figsize=(5, 6))
for bdot in [0.0, 0.05, 0.1, 0.3, 1.0]:
    Pe = bdot * H / kappa_yr
    # T = robin_profile(z, Ts, G, H, bdot)
    # ax.plot(T, z, lw=2, label=f'bdot = {bdot} m/yr, Pe = {Pe:.1f}')
    # print(f'bdot = {bdot:4.2f} m/yr   Pe = {Pe:5.1f}   basal T = {T[0]:7.1f} C')
ax.set_xlabel('temperature (deg C)')
ax.set_ylabel('height above bed (m)')
ax.legend()
plt.show()


Look at where the curvature lives. At high Peclet number the gradient survives only in a basal boundary layer of thickness roughly $H/\sqrt{\mathrm{Pe}}$, the upper column rides down nearly isothermal at the surface temperature, and the bed ends up far colder than the conduction-only estimate. Advection refrigerates the bed, because accumulation is a conveyor delivering cold surface ice downward faster than conduction can warm it. This is why the bed under the high-accumulation summit of Greenland can be frozen while the bed under the colder but nearly snowless East Antarctic plateau is at the melting point. Before moving on, confirm that ordering with your own numbers from Task 2.


## Periodic surface forcing

The surface temperature is not constant. It swings through the seasons, and through slower cycles too, and each swing launches a thermal wave downward into the ice. For a sinusoidal surface forcing of amplitude $\Delta T$ and period $P$, the heat equation admits a damped traveling wave. Writing $d$ for depth below the surface,

$$ T(d, t) = \bar T_s + \Delta T\, e^{-d/d^*} \sin\!\left(\frac{2\pi t}{P} - \frac{d}{d^*}\right), \qquad d^* = \sqrt{\frac{\kappa P}{\pi}}. $$

The amplitude falls by a factor of $e$ over each e-folding depth $d^*$, and the phase lags by one radian over the same distance, so at depth the wave arrives both weakened and late. The scaling with $\sqrt{P}$ is the signature of diffusion. Longer cycles reach deeper, but only as the square root of their period, which is why the chapter could assert that the seasonal swing is erased within the upper ten to twenty meters while leaving the deeper ice to remember the mean climate.

**Task 3:** Compute $d^*$ for the annual cycle and for a 10-year cycle. For the annual wave with $\Delta T = 10$ °C, plot the amplitude envelope $\pm\Delta T\, e^{-d/d^*}$ over the upper 25 m together with twelve monthly snapshots of the full solution, and find the depth at which the annual amplitude drops below 0.1 °C. At that depth, by how many months does the temperature maximum lag midsummer?


In [ ]:
dT_amp = 10.0     # forcing amplitude, deg C
Ts_mean = -20.0   # mean surface temperature, deg C
d = np.linspace(0.0, 25.0, 251)

for P in [1.0, 10.0]:
    d_star = np.sqrt(kappa_yr * P / np.pi)
    print(f'period {P:5.1f} yr -> e-folding depth d* = {d_star:.2f} m')

P = 1.0
d_star = np.sqrt(kappa_yr * P / np.pi)

fig, ax = plt.subplots(figsize=(5, 6))
for month in range(12):
    t = month / 12.0
    # T = Ts_mean + ...   # YOUR CODE HERE: the damped wave at time t
    # ax.plot(T, d, color='steelblue', alpha=0.5, lw=1)

# YOUR CODE HERE: overplot the envelope Ts_mean +/- dT_amp*exp(-d/d_star),
# then find the depth where the amplitude falls below 0.1 deg C and the
# phase lag (d/d_star radians, converted to months) at that depth.

ax.invert_yaxis()
ax.set_xlabel('temperature (deg C)')
ax.set_ylabel('depth below surface (m)')
plt.show()


## A transient finite-difference solver

Analytic solutions stop at steps and sinusoids. For an arbitrary climate history we discretize. Divide the column into nodes spaced $\Delta z$ apart and march forward in time with the explicit scheme

$$ T_i^{n+1} = T_i^n + \frac{\kappa\,\Delta t}{\Delta z^2}\left(T_{i+1}^n - 2T_i^n + T_{i-1}^n\right), $$

which is stable only when $\Delta t \le \Delta z^2 / 2\kappa$, the price of an explicit method. The solver below is complete. It holds the basal gradient at $-G/k$ with a ghost-node trick and lets you prescribe any surface temperature history you like. We neglect advection here, which is a fair approximation for the slow interior sites where borehole thermometry is done.

Run the demonstration, a $+2$ °C step in surface temperature applied to a 2000-m column in steady state. Watch the warming front diffuse downward over the following five thousand years, and notice the front's depth growing like $\sqrt{\kappa t}$, the same square-root law as the periodic waves.


In [ ]:
def evolve_temperature(T0, z, dt, t_end, Ts_func, G, snap_times):
    """March the 1-D conduction equation forward in time (explicit FTCS).

    T0         : initial temperature profile on the grid z, deg C
    z          : uniform grid of heights above the bed, m
    dt         : time step, yr (must satisfy dt <= dz^2 / (2 kappa))
    t_end      : total integration time, yr
    Ts_func    : function of time (yr) returning surface temperature, deg C
    G          : geothermal flux, W m^-2
    snap_times : times (yr) at which to store the profile
    """
    dz = z[1] - z[0]
    r = kappa_yr * dt / dz**2
    assert r <= 0.5, f'unstable: r = {r:.2f}, reduce dt or coarsen dz'
    T = T0.copy()
    nsteps = int(round(t_end / dt))
    snaps, recorded = {}, set()
    for step in range(nsteps + 1):
        t = step * dt
        for ts in snap_times:
            if ts not in recorded and t >= ts:
                snaps[ts] = T.copy()
                recorded.add(ts)
        Tnew = T.copy()
        Tnew[1:-1] = T[1:-1] + r * (T[2:] - 2 * T[1:-1] + T[:-2])
        ghost = T[1] + 2 * dz * (G / k)      # enforces dT/dz = -G/k at z=0
        Tnew[0] = T[0] + r * (T[1] - 2 * T[0] + ghost)
        Tnew[-1] = Ts_func(t)
        T = Tnew
    return snaps


H = 2000.0
Ts0, G = -30.0, 0.050
z = np.linspace(0.0, H, 101)
dz = z[1] - z[0]
dt = 0.4 * dz**2 / (2 * kappa_yr)

T_init = Ts0 + (G / k) * (H - z)             # conduction steady state
step_fn = lambda t: Ts0 + 2.0                # +2 C step at t = 0
snap_times = [0.0, 100.0, 500.0, 1000.0, 2000.0, 5000.0]
snaps = evolve_temperature(T_init, z, dt, 5000.0, step_fn, G, snap_times)

fig, ax = plt.subplots(figsize=(5, 6))
for ts in snap_times:
    ax.plot(snaps[ts], z, lw=2, label=f't = {ts:.0f} yr')
    print(f't = {ts:6.0f} yr   diffusion depth ~ {np.sqrt(kappa_yr * max(ts, 1e-9)):7.0f} m')
ax.set_xlabel('temperature (deg C)')
ax.set_ylabel('height above bed (m)')
ax.legend()
plt.show()


## Task 4: the basal thermal state, four ways

Now put the steady machinery to work on the question that matters. The table gives rounded parameters for four settings, a temperate-zone valley glacier and three ice-sheet sites.

| Setting | $T_s$ (°C) | $H$ (m) | $\dot b$ (m yr$^{-1}$) | $G$ (mW m$^{-2}$) |
|---|---|---|---|---|
| Cascades valley glacier | $-2$ | 200 | 3.0 | 60 |
| East Antarctic plateau (Dome C-like) | $-54$ | 3200 | 0.03 | 54 |
| Greenland summit (GRIP-like) | $-31$ | 3000 | 0.23 | 51 |
| Siple Coast ice-stream onset | $-26$ | 1000 | 0.10 | 85 |

**Task 4:** For each setting, compute the Robin basal temperature and compare it with the pressure-melting point $T_{pmp}(H)$, classifying each bed as frozen or thawed. Then, for the Siple Coast case only, vary the geothermal flux to find the threshold value at which the bed just reaches the melting point, and comment on whether the uncertainty in West Antarctic geothermal flux (measurements span roughly 60 to well over 100 mW m$^{-2}$) is large enough to flip your answer.


In [ ]:
settings = {
    'Cascades valley glacier':  dict(Ts=-2.0,  H=200.0,  bdot=3.0,  G=0.060),
    'East Antarctic plateau':   dict(Ts=-54.0, H=3200.0, bdot=0.03, G=0.054),
    'Greenland summit':         dict(Ts=-31.0, H=3000.0, bdot=0.23, G=0.051),
    'Siple Coast onset':        dict(Ts=-26.0, H=1000.0, bdot=0.10, G=0.085),
}

for name, p in settings.items():
    z = np.linspace(0.0, p['H'], 601)
    # YOUR CODE HERE: compute the Robin profile, extract the basal
    # temperature, compare with T_pmp(p['H']), and print a verdict like
    # 'Greenland summit: basal T = -11.2 C, T_pmp = -2.6 C -> frozen'
    pass

# YOUR CODE HERE: for the Siple Coast case, loop over G values and find
# the threshold geothermal flux that brings the bed to the melting point.


## Synthesis

Write a short paragraph addressing the questions below, drawing on your plots and printed numbers.

1. Accumulation never appears as a heat source, yet it changes the basal temperature by tens of degrees in your Task 4 table. Explain the physics in your own words, without equations.
2. Your Greenland summit estimate should come out a few degrees colder than the measured GRIP basal temperature of about $-9$ °C. The model neglects strain heating and downstream advection of ice from a colder, higher interior. Which neglected process would push the model toward the observation, and which away from it?
3. The Siple Coast ice streams demand a thawed bed to slide, yet your steady model puts that bed close to the freezing threshold. What does that near-coincidence suggest about why ice streams can switch on and off, and how does it connect to the thermomechanical surge cycle described in the chapter?
4. Borehole temperature profiles at GRIP and Dye 3 still carry the cold anomaly of the Little Ice Age and the warmth of the early Holocene. Using the $\sqrt{\kappa t}$ scaling you verified with the transient solver, estimate how deep each of those signals should sit today, and explain why the deeper past becomes progressively harder to read from a borehole.
